# Sensitivity Matrix — eTransform Slider Analysis

**Question:** For each eTransform character, which parameters actually move the output?

**Method:** Brute-force — varied one parameter at a time, measured L2 delta in alpha/beta/pulse_frequency outputs.

**Decision rule:**
- Score > 0.25 → show this slider in the UI
- Score < 0.05 → dead parameter, hide it (flag to Edger)
- The cliff between groups is the real threshold

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Load results
df = pd.read_csv('results/sensitivity.csv')
print(f"{len(df)} rows — {df['etransform'].nunique()} eTransforms × {df['parameter'].nunique()} parameters × {df['fixture'].nunique()} fixtures")
df.head()

## 1. Max delta per (eTransform, parameter)
Take the max delta_mean across all step values and fixtures. This is the 'ceiling' — the most this parameter can move the output.

In [ ]:
# Max delta per etransform × parameter (across all values and fixtures)
max_delta = (
    df.groupby(['etransform', 'parameter'])['delta_mean']
    .max()
    .reset_index()
    .rename(columns={'delta_mean': 'max_delta'})
)

# Normalize per eTransform: top mover = 1.0
max_delta['score'] = max_delta.groupby('etransform')['max_delta'].transform(
    lambda x: x / x.max() if x.max() > 0 else x
)

max_delta.sort_values(['etransform', 'score'], ascending=[True, False])

## 2. Heatmap — the cliff is visible here

In [ ]:
pivot = max_delta.pivot(index='parameter', columns='etransform', values='score').fillna(0)

# Sort rows by mean score descending
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

# Threshold lines
ax.axhline(y=sum(pivot.mean(axis=1) > 0.25) - 0.5, color='blue', linewidth=2, linestyle='--', label='25% threshold')

plt.colorbar(im, ax=ax, label='Normalized score (1.0 = biggest mover per eTransform)')
ax.set_title('Parameter Sensitivity by eTransform\n(higher = more impact on output)', fontsize=13)
plt.tight_layout()
plt.savefig('results/sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/sensitivity_heatmap.png')

## 3. The verdict — what to show in the UI

In [ ]:
SHOW_THRESHOLD = 0.25
DEAD_THRESHOLD = 0.05

print('=== SHOW IN UI (score > 0.25) ===')
show = max_delta[max_delta['score'] > SHOW_THRESHOLD].sort_values(['etransform', 'score'], ascending=[True, False])
for et, group in show.groupby('etransform'):
    params = ', '.join(f"{r['parameter']} ({r['score']:.2f})" for _, r in group.iterrows())
    print(f"  {et}: {params}")

print()
print('=== DEAD PARAMETERS (score < 0.05 across all eTransforms) ===')
dead = max_delta.groupby('parameter')['score'].max()
dead_params = dead[dead < DEAD_THRESHOLD].index.tolist()
for p in dead_params:
    print(f'  {p}')
if not dead_params:
    print('  None — all parameters affect at least one eTransform')

## 4. Check for surprises
Parameters we didn't expect to matter that score high.

In [ ]:
EXPECTED_SLIDERS = {
    'Gentle':        ['alpha_beta_generation.min_distance_from_center', 'pulse.pulse_rise_max'],
    'Reactive':      ['frequency.frequency_ramp_combine_ratio', 'frequency.pulse_freq_max'],
    'Scene Builder': ['frequency.frequency_ramp_combine_ratio', 'alpha_beta_generation.min_distance_from_center'],
    'Unpredictable': ['alpha_beta_generation.min_distance_from_center', 'frequency.pulse_frequency_combine_ratio'],
    'Balanced':      ['alpha_beta_generation.min_distance_from_center', 'frequency.frequency_ramp_combine_ratio'],
}

print('=== SURPRISES — high-scoring parameters not in our current UI choices ===')
for et, expected in EXPECTED_SLIDERS.items():
    subset = max_delta[(max_delta['etransform'] == et) & (max_delta['score'] > SHOW_THRESHOLD)]
    surprises = subset[~subset['parameter'].isin(expected)]
    if not surprises.empty:
        for _, r in surprises.iterrows():
            print(f"  [{et}] SURPRISE: {r['parameter']} scores {r['score']:.2f} but not in current UI")

print()
print('=== MISSES — our current UI choices that score low ===')
for et, expected in EXPECTED_SLIDERS.items():
    for param in expected:
        row = max_delta[(max_delta['etransform'] == et) & (max_delta['parameter'] == param)]
        if row.empty or row['score'].values[0] < SHOW_THRESHOLD:
            score = row['score'].values[0] if not row.empty else 0.0
            print(f"  [{et}] MISS: {param} scores {score:.2f} — consider replacing")

## 5. Consistency across fixtures
Does a parameter behave the same on raw vs. forged content?

In [ ]:
if df['fixture'].nunique() > 1:
    consistency = (
        df.groupby(['etransform', 'parameter', 'fixture'])['delta_mean']
        .max()
        .reset_index()
    )
    # Pivot by fixture
    pivot_fix = consistency.pivot_table(
        index=['etransform', 'parameter'], columns='fixture', values='delta_mean'
    ).reset_index()
    
    fixture_cols = [c for c in pivot_fix.columns if c not in ('etransform', 'parameter')]
    if len(fixture_cols) == 2:
        a, b = fixture_cols
        pivot_fix['ratio'] = pivot_fix[a] / (pivot_fix[b] + 1e-9)
        # Highly content-dependent = ratio far from 1.0
        content_dependent = pivot_fix[
            (pivot_fix['ratio'] > 3) | (pivot_fix['ratio'] < 0.33)
        ].sort_values('ratio', ascending=False)
        print('Content-dependent parameters (behave very differently on raw vs forged):')
        print(content_dependent[['etransform', 'parameter', a, b, 'ratio']].to_string(index=False))
else:
    print('Only one fixture — run with both raw and forged for consistency check')

## 6. Updated slider recommendations
What the data says to show, replacing our educated guesses.

In [ ]:
print('=== RECOMMENDED SLIDER CHOICES (top 2 per eTransform) ===')
print('Copy these into BUILTIN_PRESETS in cli.py')
print()
for et in BUILTIN_PRESETS_ORDER := ['Gentle', 'Reactive', 'Scene Builder', 'Unpredictable', 'Balanced']:
    subset = max_delta[max_delta['etransform'] == et].sort_values('score', ascending=False)
    top2 = subset.head(2)
    params = [f"{r['parameter']} (score={r['score']:.2f})" for _, r in top2.iterrows()]
    print(f"  {et}:")
    for p in params:
        print(f"    {p}")